# Stage 3. Compute background functions

## 1. Setup

### 1.1 Imports

In [ ]:
import datetime
import logging
from collections.abc import Sequence
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

from acid.config import load_config, prepare_output
from acid.image_processing.calculate_background_function import (
    calculate_background_function,
    compute_simple_background,
    get_polyfit_bg_funct_channel,
    import_fov,
)
from acid.utils.fov_axis_utils import get_fov_ch_shape
from acid.utils.metadata.filtering import filter_metadata_by_splits
from acid.utils.metadata.loading import load_metadata
from acid.utils.save_image import tifffile_save_ometiff

### 1.2 Configure logging

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

### 1.3 Load configuration

In [ ]:
CONFIG_FILE = None  # "configs/exp_001.yaml"
config = load_config(CONFIG_FILE)
prepare_output(config)
stage_cfg = config.background_function_calculation
paths_cfg = config.shared.paths

## 2. Input Data Preparation

In [ ]:
metadata_df, metadata_file_name = load_metadata(stage_cfg.metadata)
metadata_df = filter_metadata_by_splits(
    metadata_df, selected_splits="train", split_config=stage_cfg.dataset_split
).copy()
flag_column = stage_cfg.metadata.dataframe_columns.flag_column_name
metadata_df_ok = metadata_df.loc[
    metadata_df[flag_column] != config.quality_control.processing.flag_value
].copy()
if metadata_df_ok.empty:
    raise ValueError("No unflagged training images available for background estimation")
fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(
    metadata_df_ok,
    stage_cfg.processing.fov_directory,
    fov_clm=stage_cfg.metadata.dataframe_columns.fov_column_name,
    channel_axis=stage_cfg.processing.channel_axis,
    null_value=stage_cfg.metadata.dataframe_columns.null_value,
)
Path(stage_cfg.image_saving.directory).mkdir(parents=True, exist_ok=True)

## 3. Background calculation
### 3.1 Helpers

In [ ]:
# FUTURE: move to src/acid/image_processing/background/calculate_background.py
def calculate_backgrounds(metadata_df, fov_shape, cfg):
    proc = cfg.processing
    columns = cfg.metadata.dataframe_columns
    strategy = proc.background_function_strategy
    if strategy == 1:
        groups = [(None, metadata_df)]
    elif strategy in (2, 3):
        column = (
            columns.well_column_name if strategy == 2 else columns.gridpos_column_name
        )
        if metadata_df[column].isna().any():
            raise ValueError(f"Missing background grouping values in {column}")
        groups = metadata_df.groupby(column, sort=False)
    else:
        raise ValueError("background_function_strategy must be 1, 2 or 3")
    if proc.axis_calc_bg not in (-1, len(fov_shape)):
        raise ValueError("axis_calc_bg must select the appended image-stack axis")
    backgrounds = {}
    for key, rows in groups:
        stack = import_fov(
            df=rows,
            fov_dir=proc.fov_directory,
            fov_clm=columns.fov_column_name,
            fov_shape=fov_shape,
            verbose=proc.verbose_calc_bg,
            np_zero_kwargs=OmegaConf.to_container(proc.np_zero_kwargs)
            if proc.np_zero_kwargs is not None
            else None,
            sample_df=proc.sample_df,
            sample_fraction=proc.sample_fraction,
            sample_kwargs=OmegaConf.to_container(proc.sample_kwargs),
        )
        if stack.shape[-1] == 0:
            raise ValueError(f"Sampling selected no images for background group {key}")
        backgrounds[key] = calculate_background_function(
            stack,
            method=proc.background_function_avg_method,
            axis=-1,
            verbose=proc.verbose_calc_bg,
        )
    return backgrounds


# FUTURE: move to src/acid/image_processing/background/calculate_background.py
def fit_background(background, cfg):
    proc = cfg.processing
    if proc.background_fit_method == "polyfit":
        return get_polyfit_bg_funct_channel(
            background_function=background,
            channel_axis=proc.channel_axis,
            kx=tuple(proc.polynomial_order_x),
            ky=tuple(proc.polynomial_order_y),
            verbose=proc.verbose_calc_bg,
        )
    if proc.background_fit_method != "simple":
        raise ValueError("background_fit_method must be polyfit or simple")
    params = OmegaConf.to_container(proc, resolve=True)
    return compute_simple_background(
        background,
        ball_radius=params["ball_radius"],
        white_background=params["white_background"],
        _rb_kwargs=params["rolling_ball_kwargs"],
        invert_kwargs=params["invert_kwargs"],
        gau_smooth=params["gau_smooth"],
        gaussian_kwargs=params["gaussian_kwargs"],
        convolve_kwargs=params["convolve_kwargs"],
        dtype=params["dtype"],
        axis=proc.channel_axis,
        n_workers=proc.n_workers,
        map_kwargs=params["map_kwargs"],
    )


# FUTURE: move to src/acid/image_processing/background/save_background.py
def save_backgrounds(backgrounds, cfg, project_name):
    proc, saving, meta = cfg.processing, cfg.image_saving, cfg.metadata.image_metadata
    output = Path(saving.directory)
    output.mkdir(parents=True, exist_ok=True)
    saved = []
    for key, background in backgrounds.items():
        fitted = fit_background(background, cfg)
        image_metadata = {
            meta.background_img_meta_date_name: datetime.datetime.now()
            .astimezone()
            .strftime(meta.background_img_meta_date_format),
            meta.background_img_meta_project_name: project_name,
            meta.background_img_meta_avg_method_name: proc.background_function_avg_method,
        }
        if proc.background_fit_method == "polyfit":
            image_metadata[meta.background_img_meta_poly_degree_name] = (
                f"kx: {list(proc.polynomial_order_x)}, ky: {list(proc.polynomial_order_y)}, order: None"
            )
        else:
            params = OmegaConf.to_container(proc, resolve=True)
            image_metadata.update(
                {
                    meta.background_img_meta_ballradius_name: params["ball_radius"],
                    meta.background_img_meta_whitebg_name: params["white_background"],
                    meta.background_img_meta_gausmooth_name: params["gau_smooth"],
                }
            )
        if key is not None:
            image_metadata[
                "well" if proc.background_function_strategy == 2 else "grid_position"
            ] = str(key)
        parts = [
            datetime.datetime.now()
            .astimezone()
            .strftime(saving.background_img_name_date_format),
            project_name,
            saving.background_img_savingword,
        ]
        if key is not None:
            parts.append(str(key))
        filename = (
            saving.save_file_name_separator.join(parts)
            + saving.background_img_file_suffix
        )
        path = output / filename
        tifffile_save_ometiff(
            str(path),
            data=fitted.astype(saving.background_img_dtype),
            imagej=saving.save_imagej_compatible,
            photometric=saving.photometric,
            metadata=image_metadata,
        )
        if saving.save_avg_background_function_image:
            directory = Path(saving.average_background_directory)
            directory.mkdir(parents=True, exist_ok=True)
            parts[2] = saving.non_polyfit_background_img_savingword
            average_path = directory / (
                saving.save_file_name_separator.join(parts)
                + saving.background_img_file_suffix
            )
            tifffile_save_ometiff(
                str(average_path),
                data=background.astype(saving.background_img_dtype),
                imagej=saving.save_imagej_compatible,
                photometric=saving.photometric,
                metadata={
                    meta.background_img_meta_avg_method_name: proc.background_function_avg_method
                },
            )
        saved.append({"group": key, "output_file": str(path), "shape": fitted.shape})
    return saved


# FUTURE: move to src/acid/image_processing/background/metadata.py
def update_background_metadata(metadata_df, cfg):
    updated = metadata_df.copy()
    proc, columns = cfg.processing, cfg.metadata.dataframe_columns
    updated[columns.background_df_date_clm_name] = (
        datetime.datetime.now()
        .astimezone()
        .strftime(columns.background_df_meta_date_format)
    )
    updated[columns.background_df_avg_method_clm_name] = (
        proc.background_function_avg_method
    )
    params = OmegaConf.to_container(proc, resolve=True)
    if proc.background_fit_method == "polyfit":
        fields = [
            (columns.background_df_poly_order_x_clm_name, params["polynomial_order_x"]),
            (columns.background_df_poly_order_y_clm_name, params["polynomial_order_y"]),
        ]
    else:
        fields = [
            (columns.background_df_meta_ballradius_name, params["ball_radius"]),
            (columns.background_df_meta_whitebg_name, params["white_background"]),
            (columns.background_df_meta_gausmooth_name, params["gau_smooth"]),
        ]
    for name, value in fields:
        if isinstance(value, Sequence) and not isinstance(value, (str, bytes)):
            for channel, item in enumerate(value):
                updated[f"{name}{columns.channel_name_separator}{channel}"] = item
        else:
            updated[name] = value
    return updated

### 3.2 Calculate and save the selected strategy

In [ ]:
backgrounds = calculate_backgrounds(metadata_df_ok, fov_shape, stage_cfg)
results = save_backgrounds(backgrounds, stage_cfg, config.project_identity.project_name)
metadata_df_updated = update_background_metadata(metadata_df, stage_cfg)
pd.DataFrame(results)

## 4. Save metadata

In [ ]:
# FUTURE: move to src/acid/utils/metadata/saving.py
def save_metadata_dataframe(metadata_df, metadata_config, project_name):
    saving = metadata_config.saving
    separator = saving.save_file_name_separator
    suffix = saving.metadata_file_suffix.format(save_file_name_separator=separator)
    filename = separator.join(
        [
            datetime.datetime.now().astimezone().strftime(saving.metadata_date_format),
            project_name,
            saving.metadata_savingword,
            suffix,
        ]
    )
    path = Path(metadata_config.directory) / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    metadata_df.to_csv(path, index=saving.save_csv_index)
    logging.getLogger(__name__).info("Saved metadata: %s", path)
    return path


metadata_path = save_metadata_dataframe(
    metadata_df_updated, stage_cfg.metadata, config.project_identity.project_name
)

## 5. Inspect results

In [ ]:
metadata_df_updated.head()

### 5.1 Display average backgrounds

In [ ]:
for group, background in backgrounds.items():
    fig, axes = plt.subplots(
        1, num_channels, figsize=(4 * num_channels, 4), squeeze=False
    )
    for channel, ax in enumerate(axes[0]):
        ax.imshow(
            np.take(background, channel, axis=stage_cfg.processing.channel_axis),
            cmap="gray",
        )
        ax.set_title(f"{group}: channel {channel}")
    plt.show()